<a href="https://colab.research.google.com/github/emqzhang/phi2-finetuning/blob/main/Phi2_QLoRA_Comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phi-2 Dialogue Summarization: LoRA Target-Module Ablation

**Research question:** Does adapting additional attention/output projections improve dialogue summarization enough to justify the extra trainable parameters and training time?

This notebook runs a 4-bit Phi-2 baseline and two LoRA variants on the same held-out examples.

## 1. Setup and data

Requires a CUDA GPU and compatible `bitsandbytes` environment.

In [ ]:
!pip install -U bitsandbytes transformers peft accelerate datasets scipy einops evaluate trl rouge_score -qq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 6.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 113.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 17.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarro

In [ ]:
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    GenerationConfig
)
from tqdm import tqdm
from trl import SFTTrainer
import torch
import time
import pandas as pd
import numpy as np
# from huggingface_hub import interpreter_login

# interpreter_login()

In [ ]:
import os
# disable Weights and Biases
os.environ['WANDB_DISABLED']="true"

In [ ]:
from pynvml import *

def print_gpu_utilization():
    nvmlInit()
    handle = nvmlDeviceGetHandleByIndex(0)
    info = nvmlDeviceGetMemoryInfo(handle)
    print(f"GPU memory occupied: {info.used//1024**2} MB.")

# Load dataset

In [ ]:
huggingface_dataset_name = "neil-code/dialogsum-test"
dataset = load_dataset(huggingface_dataset_name)

README.md:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/1.81M [00:00<?, ?B/s]

validation.csv:   0%|          | 0.00/441k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/447k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1999 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/499 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/499 [00:00<?, ? examples/s]

In [ ]:
dataset['train'][0]

{'id': 'train_0',
 'dialogue': "#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?\n#Person2#: I found it would be a good idea to get a check-up.\n#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.\n#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?\n#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.\n#Person2#: Ok.\n#Person1#: Let me see here. Your eyes and ears look fine. Take a deep breath, please. Do you smoke, Mr. Smith?\n#Person2#: Yes.\n#Person1#: Smoking is the leading cause of lung cancer and heart disease, you know. You really should quit.\n#Person2#: I've tried hundreds of times, but I just can't seem to kick the habit.\n#Person1#: Well, we have classes and some medications that might help. I'll give you more information before you leave.\n#Person2#: Ok, thanks doctor.",
 'summary': "Mr. Smith'

In [ ]:
compute_dtype = getattr(torch, "float16")
bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=False,
    )

# Load base model

In [ ]:
model_name='microsoft/phi-2'
device_map = {"": 0}
original_model = AutoModelForCausalLM.from_pretrained(model_name,
                                                      device_map=device_map,
                                                      quantization_config=bnb_config,
                                                      trust_remote_code=True,
                                                      )

tokenizer = AutoTokenizer.from_pretrained(model_name,
                                          trust_remote_code=True,
                                          padding_side="left",
                                          add_eos_token=True,add_bos_token=True,
                                          use_fast=False)
tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.7k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

...

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/1.08k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

In [ ]:
eval_tokenizer = AutoTokenizer.from_pretrained(model_name, add_bos_token=True, trust_remote_code=True, use_fast=False)
eval_tokenizer.pad_token = eval_tokenizer.eos_token

def gen(model,p, maxlen=100, sample=True):
    toks = eval_tokenizer(p, return_tensors="pt")
    res = model.generate(**toks.to("cuda"), max_new_tokens=maxlen, do_sample=sample,num_return_sequences=1,temperature=0.1,num_beams=1,top_p=0.95,).to('cpu')
    return eval_tokenizer.batch_decode(res,skip_special_tokens=True)

# Preprocess dataset

In [ ]:
def create_prompt_formats(sample):
    """
    Format various fields of the sample ('instruction','output')
    Then concatenate them using two newline characters
    :param sample: Sample dictionnary
    """
    INTRO_BLURB = "Below is an instruction that describes a task. Write a response that appropriately completes the request."
    INSTRUCTION_KEY = "### Instruct: Summarize the below conversation."
    RESPONSE_KEY = "### Output:"
    END_KEY = "### End"

    blurb = f"\n{INTRO_BLURB}"
    instruction = f"{INSTRUCTION_KEY}"
    input_context = f"{sample['dialogue']}" if sample["dialogue"] else None
    response = f"{RESPONSE_KEY}\n{sample['summary']}"
    end = f"{END_KEY}"

    parts = [part for part in [blurb, instruction, input_context, response, end] if part]

    formatted_prompt = "\n\n".join(parts)
    sample["text"] = formatted_prompt

    return sample

In [ ]:
# SOURCE https://github.com/databrickslabs/dolly/blob/master/training/trainer.py
def get_max_length(model):
    conf = model.config
    max_length = None
    for length_setting in ["n_positions", "max_position_embeddings", "seq_length"]:
        max_length = getattr(model.config, length_setting, None)
        if max_length:
            print(f"Found max lenth: {max_length}")
            break
    if not max_length:
        max_length = 1024
        print(f"Using default max length: {max_length}")
    return max_length


def preprocess_batch(batch, tokenizer, max_length):
    """
    Tokenizing a batch
    """
    return tokenizer(
        batch["text"],
        max_length=max_length,
        truncation=True,
    )

In [ ]:
from functools import partial

# SOURCE https://github.com/databrickslabs/dolly/blob/master/training/trainer.py
def preprocess_dataset(tokenizer: AutoTokenizer, max_length: int,seed, dataset):
    """Format & tokenize it so it is ready for training
    :param tokenizer (AutoTokenizer): Model Tokenizer
    :param max_length (int): Maximum number of tokens to emit from tokenizer
    """

    # Add prompt to each sample
    print("Preprocessing dataset...")
    dataset = dataset.map(create_prompt_formats)#, batched=True)

    _preprocessing_function = partial(preprocess_batch, max_length=max_length, tokenizer=tokenizer)
    dataset = dataset.map(
        _preprocessing_function,
        batched=True,
        remove_columns=['id', 'topic', 'dialogue', 'summary'],
    )

    # Filter out samples that have input_ids exceeding max_length
    dataset = dataset.filter(lambda sample: len(sample["input_ids"]) < max_length)

    # Shuffle dataset
    dataset = dataset.shuffle(seed=seed)

    return dataset

In [ ]:
# ## Pre-process dataset
max_length = get_max_length(original_model)
seed = 67
print(max_length)

train_dataset = preprocess_dataset(tokenizer, max_length, seed, dataset['train'])
eval_dataset = preprocess_dataset(tokenizer, max_length, seed, dataset['validation'])

Found max lenth: 2048
2048
Preprocessing dataset...


Map:   0%|          | 0/1999 [00:00<?, ? examples/s]

Map:   0%|          | 0/1999 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1999 [00:00<?, ? examples/s]

Preprocessing dataset...


Map:   0%|          | 0/499 [00:00<?, ? examples/s]

Map:   0%|          | 0/499 [00:00<?, ? examples/s]

Filter:   0%|          | 0/499 [00:00<?, ? examples/s]

In [ ]:
print(f"Shapes of the datasets:")
print(f"Training: {train_dataset.shape}")
print(f"Validation: {eval_dataset.shape}")
print(train_dataset)

Shapes of the datasets:
Training: (1999, 3)
Validation: (499, 3)
Dataset({
    features: ['text', 'input_ids', 'attention_mask'],
    num_rows: 1999
})


In [ ]:
train_dataset['text'][1]

"\nBelow is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruct: Summarize the below conversation.\n\n#Person1#: I haven't told you what happened yet, have I? \n#Person2#: I haven't heard anything. \n#Person1#: My boss offered me a promotion, and I took it. \n#Person2#: Are you serious? \n#Person1#: Yes, I am really excited. \n#Person2#: That's great. Congratulations. \n#Person1#: I appreciate that. \n#Person2#: You have no idea how happy I am for you. \n#Person1#: For real? \n#Person2#: I believe you were the best choice for that promotion. I really do. \n\n### Output:\n#Person1# tells #Person2# #Person1#'s promotion. #Person2# believes #Person1#'s the best choice for it.\n\n### End"

## 2. Experiment configuration

Keep the seed, data preprocessing, training budget, and generation routine fixed. Only LoRA target modules vary. Both variants use rank 32, alpha 32, and dropout 0.05.

In [ ]:

def print_number_of_trainable_model_parameters(model):
    trainable_model_params = 0
    all_model_params = 0
    for _, param in model.named_parameters():
        all_model_params += param.numel()
        if param.requires_grad:
            trainable_model_params += param.numel()
    percentage = 100 * trainable_model_params / all_model_params
    summary = (
        f"trainable model parameters: {trainable_model_params}\n"
        f"all model parameters: {all_model_params}\n"
        f"percentage of trainable model parameters: {percentage:.4f}%"
    )
    return summary, trainable_model_params, all_model_params, percentage


## 3. Shared model, trainer, and evaluation functions

`run_lora_experiment` saves adapter checkpoints and per-run JSON results. ROUGE measures n-gram overlap with reference summaries; it is not a factuality or human-preference metric.

In [ ]:

# Experiment controls
base_model_id = "microsoft/phi-2"
SEED = 42
MAX_STEPS = 500          #changed from 1000 to 500
EVAL_SAMPLE_SIZE = 10
BASE_OUTPUT_DIR = "./phi2_lora_outputs"

os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)

DEFAULT_TARGET_MODULES = ['q_proj', 'k_proj', 'v_proj', 'dense']
QV_ONLY_TARGET_MODULES = ['q_proj', 'v_proj']

EXPERIMENTS = {
    "default_lora": DEFAULT_TARGET_MODULES,
    "qv_only_lora": QV_ONLY_TARGET_MODULES,
}


In [ ]:
import time
import pandas as pd

In [ ]:
import transformers

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
import evaluate
import json

def load_quantized_base_model(model_id=base_model_id):
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto",
        quantization_config=bnb_config,
        trust_remote_code=True,

    )
    return model

def build_peft_model(target_modules, model_id=base_model_id):
    model = load_quantized_base_model(model_id)
    model.gradient_checkpointing_enable()
    model = prepare_model_for_kbit_training(model)

    config = LoraConfig(
        r=32,
        lora_alpha=32,
        target_modules=target_modules,
        bias="none",
        lora_dropout=0.05,
        task_type="CAUSAL_LM",
    )

    peft_model = get_peft_model(model, config)
    return peft_model, config

def build_trainer(model, output_dir):
    training_args = TrainingArguments(
        output_dir=output_dir,
        warmup_steps=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        max_steps=MAX_STEPS,
        learning_rate=2e-4,
        optim="paged_adamw_8bit",
        logging_steps=25,
        save_strategy="steps",
        save_steps=50,
        eval_strategy="steps",
        eval_steps=50,
        do_eval=True,
        gradient_checkpointing=True,
        report_to="none",
        seed=SEED,
    )

    model.config.use_cache = False

    trainer = Trainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        args=training_args,
        data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
    )
    return trainer, training_args

def generate_summary(model, dialogue, max_new_tokens=100):
    prompt = f"Instruct: Summarize the following conversation.\n{dialogue}\nOutput:\n"
    response = gen(model, prompt, max_new_tokens)[0]
    output = response.split("Output:\n", 1)[1] if "Output:\n" in response else response
    output = output.split("### End", 1)[0]
    output = output.split("#End", 1)[0]
    return output.strip()

def evaluate_model_rouge(model, dataset_obj, sample_size=EVAL_SAMPLE_SIZE):
    dialogues = dataset_obj['test'][0:sample_size]['dialogue']
    references = dataset_obj['test'][0:sample_size]['summary']

    predictions = []
    for dialogue in dialogues:
        predictions.append(generate_summary(model, dialogue))

    rouge = evaluate.load("rouge")
    rouge_scores = rouge.compute(
        predictions=predictions,
        references=references,
        use_aggregator=True,
        use_stemmer=True,
    )
    return predictions, references, rouge_scores

def run_lora_experiment(experiment_name, target_modules):
    output_dir = os.path.join(BASE_OUTPUT_DIR, experiment_name)

    peft_model, lora_config = build_peft_model(target_modules)
    param_summary, trainable_params, all_params, trainable_pct = print_number_of_trainable_model_parameters(peft_model)
    print(param_summary)

    trainer, training_args = build_trainer(peft_model, output_dir)
    print(f"Training device: {training_args.device}")

    start_time = time.time()
    train_result = trainer.train()
    fine_tune_seconds = time.time() - start_time

    adapter_path = trainer.state.best_model_checkpoint or trainer.state.global_step
    trainer.save_model(output_dir)

    del trainer
    torch.cuda.empty_cache()

    base_model_for_eval = load_quantized_base_model()
    ft_model = PeftModel.from_pretrained(
        base_model_for_eval,
        output_dir,
        torch_dtype=torch.float16,
        is_trainable=False
    )

    predictions, references, rouge_scores = evaluate_model_rouge(ft_model, dataset, sample_size=EVAL_SAMPLE_SIZE)

    results = {
        "experiment_name": experiment_name,
        "target_modules": target_modules,
        "trainable_params": int(trainable_params),
        "all_params": int(all_params),
        "trainable_pct": float(trainable_pct),
        "fine_tune_seconds": float(fine_tune_seconds),
        "fine_tune_minutes": float(fine_tune_seconds / 60.0),
        "max_steps": int(MAX_STEPS),
        "eval_sample_size": int(EVAL_SAMPLE_SIZE),
        "rouge": rouge_scores,
        "output_dir": output_dir,
    }

    with open(os.path.join(output_dir, "results.json"), "w") as f:
        json.dump(results, f, indent=2)

    return ft_model, results, predictions, references


## 4. Evaluate the untuned baseline

Generate the same 10 test examples for every model. A fresh run can differ slightly from historical measurements due to sampling, environment, and inference behavior.

In [ ]:
#load vanilla phi2 model
original_model = load_quantized_base_model()
original_model_predictions, original_references, original_model_results = evaluate_model_rouge(
    original_model,
    dataset,
    sample_size=EVAL_SAMPLE_SIZE
)

print("ORIGINAL MODEL:")
print(original_model_results)


Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer CodeGenTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


ORIGINAL MODEL:
{'rouge1': np.float64(0.27994597666763726), 'rouge2': np.float64(0.09913062746652335), 'rougeL': np.float64(0.20295230554001203), 'rougeLsum': np.float64(0.21477656631434217)}


## 5. Experiment A — broader LoRA targets

Train adapters for query, key, value, and dense projection layers (`q_proj`, `k_proj`, `v_proj`, `dense`). This increases trainable capacity relative to Q/V-only LoRA.

In [ ]:
import time
start_time = time.time()

default_ft_model, default_results, default_predictions, default_references = run_lora_experiment(
    "default_lora",
    DEFAULT_TARGET_MODULES
)

print(json.dumps(default_results, indent=2))


end_time = time.time()
print(f"Total training time: {end_time - start_time:.2f} seconds") #record finetuning time


Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

trainable model parameters: 20971520
all model parameters: 1542364160
percentage of trainable model parameters: 1.3597%
Training device: cuda:0


Step,Training Loss,Validation Loss
50,1.379985,1.364099
100,1.310908,1.347070
150,1.363523,1.341504
200,1.392669,1.336268
250,1.385273,1.332265
300,1.329946,1.331218
350,1.393203,1.330791
400,1.370033,1.328008
450,1.341563,1.326805
500,1.341967,1.326092


Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

{
  "experiment_name": "default_lora",
  "target_modules": [
    "q_proj",
    "k_proj",
    "v_proj",
    "dense"
  ],
  "trainable_params": 20971520,
  "all_params": 1542364160,
  "trainable_pct": 1.3596996444730667,
  "fine_tune_seconds": 2808.197762966156,
  "fine_tune_minutes": 46.803296049435936,
  "max_steps": 500,
  "eval_sample_size": 10,
  "rouge": {
    "rouge1": 0.4343175254597963,
    "rouge2": 0.1546067406260055,
    "rougeL": 0.36211976122372835,
    "rougeLsum": 0.3619277763959706
  },
  "output_dir": "./phi2_lora_outputs/default_lora"
}
Total training time: 2884.86 seconds


## 6. Experiment B — Q/V-only LoRA

Restrict adapters to query and value projections (`q_proj`, `v_proj`) to test whether a smaller adapter can retain most of the gains.

In [ ]:
import time
start_time = time.time()

qv_ft_model, qv_results, qv_predictions, qv_references = run_lora_experiment(
    "qv_only_lora",
    QV_ONLY_TARGET_MODULES
)

print(json.dumps(qv_results, indent=2))


end_time = time.time()
print(f"Total training time: {end_time - start_time:.2f} seconds") #record finetuning time



Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

trainable model parameters: 10485760
all model parameters: 1531878400
percentage of trainable model parameters: 0.6845%
Training device: cuda:0


Step,Training Loss,Validation Loss
50,1.394499,1.366198
100,1.315951,1.351103
150,1.363517,1.344743
200,1.395217,1.341982
250,1.389293,1.336593
300,1.335255,1.334163
350,1.394322,1.332700
400,1.372021,1.330433
450,1.344613,1.330016
500,1.344848,1.329285


Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

{
  "experiment_name": "qv_only_lora",
  "target_modules": [
    "q_proj",
    "v_proj"
  ],
  "trainable_params": 10485760,
  "all_params": 1531878400,
  "trainable_pct": 0.6845034175036347,
  "fine_tune_seconds": 2701.2204444408417,
  "fine_tune_minutes": 45.020340740680695,
  "max_steps": 500,
  "eval_sample_size": 10,
  "rouge": {
    "rouge1": 0.4189223119463539,
    "rouge2": 0.12702210159061744,
    "rougeL": 0.31263164830536544,
    "rougeLsum": 0.31427714984172983
  },
  "output_dir": "./phi2_lora_outputs/qv_only_lora"
}
Total training time: 2789.94 seconds


## 7. Unified model comparison

The following tables use compare model performance. Absolute ROUGE gains are score-point differences; relative gains divide by the baseline score. The baseline is not fine-tuned, so its training time is zero. The bootstrap aggregation seed is reset for each model; this affects score aggregation only and does not regenerate summaries.

In [ ]:
import os, json
import numpy as np
import pandas as pd
from IPython.display import display
from rouge_score import rouge_scorer, scoring

required = ["dataset", "EVAL_SAMPLE_SIZE", "BASE_OUTPUT_DIR", "original_model_predictions",
            "original_references", "default_predictions", "default_references", "default_results",
            "qv_predictions", "qv_references", "qv_results"]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError("Finish the vanilla/default/QV evaluation cells in this live runtime first. "
                       "Missing variables: " + ", ".join(missing))

references = list(original_references)
test_rows = dataset["test"][:EVAL_SAMPLE_SIZE]
assert references == list(test_rows["summary"]), "Baseline references differ from selected test rows."
assert references == list(default_references) == list(qv_references), "Models used different references."
assert len(references) == EVAL_SAMPLE_SIZE > 0
predictions_by_model = {
    "vanilla_phi2": list(original_model_predictions),
    "default_lora": list(default_predictions),
    "qv_only_lora": list(qv_predictions),
}
for name, predictions in predictions_by_model.items():
    assert len(predictions) == len(references), f"{name}: prediction/reference count mismatch"
    assert all(isinstance(p, str) and p.strip() for p in predictions), f"{name}: empty or invalid prediction"

# Same stemmed ROUGE and bootstrap mid-point aggregation used by evaluate's ROUGE metric.
METRICS = ("rouge1", "rouge2", "rougeL", "rougeLsum")
scorer = rouge_scorer.RougeScorer(METRICS, use_stemmer=True)
def recompute_rouge(predictions, references, bootstrap_seed=42):
    aggregator = scoring.BootstrapAggregator()
    for prediction, reference in zip(predictions, references):
        aggregator.add_scores(scorer.score(reference, prediction))
    previous_rng = np.random.get_state()
    try:
        np.random.seed(bootstrap_seed)
        aggregated = aggregator.aggregate()
    finally:
        np.random.set_state(previous_rng)
    return {metric: float(aggregated[metric].mid.fmeasure) for metric in METRICS}

recomputed_scores = {name: recompute_rouge(predictions, references)
                     for name, predictions in predictions_by_model.items()}
# Replace score dictionaries so later cells cannot accidentally reuse stale values.
original_model_results = recomputed_scores["vanilla_phi2"]
default_results["rouge"] = recomputed_scores["default_lora"]
qv_results["rouge"] = recomputed_scores["qv_only_lora"]
rouge_df = pd.DataFrame([{"model": name, **scores} for name, scores in recomputed_scores.items()])
print("ROUGE recomputed from current unified-run predictions:")
display(rouge_df.round(6))

ROUGE recomputed from current unified-run predictions:


,model,rouge1,rouge2,rougeL,rougeLsum
0,vanilla_phi2,0.279663,0.099322,0.202289,0.213901
1,default_lora,0.434318,0.154607,0.362120,0.361928
2,qv_only_lora,0.418922,0.127022,0.312632,0.314277


In [ ]:
# default vs vanilla
#Q/V vs vanilla and default.
model_info = {
    "vanilla_phi2": {"trainable_params": 0, "fine_tune_seconds": 0.0, "target_modules": []},
    "default_lora": default_results,
    "qv_only_lora": qv_results,
}
rows = []
for name, info in model_info.items():
    rows.append({"model": name, "parameters_updated": int(info["trainable_params"]),
                 "fine_tune_seconds": float(info["fine_tune_seconds"]),
                 "fine_tune_minutes": float(info["fine_tune_seconds"]) / 60,
                 "target_modules": ", ".join(info["target_modules"]) or "none",
                 **recomputed_scores[name]})
metrics_df = pd.DataFrame(rows)

gain_rows = []
for candidate, comparator in [("default_lora", "vanilla_phi2"),
                              ("qv_only_lora", "vanilla_phi2"),
                              ("qv_only_lora", "default_lora")]:
    for metric in METRICS:
        score = recomputed_scores[candidate][metric]
        base = recomputed_scores[comparator][metric]
        gain_rows.append({"model": candidate, "compared_with": comparator, "metric": metric,
                          "score": score, "comparator_score": base, "absolute_delta": score-base,
                          "relative_delta_pct": 100*(score-base)/base if base else np.nan})
gains_df = pd.DataFrame(gain_rows)

default_params = int(default_results["trainable_params"])
qv_params = int(qv_results["trainable_params"])
default_time = float(default_results["fine_tune_seconds"])
qv_time = float(qv_results["fine_tune_seconds"])
efficiency_df = pd.DataFrame([{
    "comparison": "QV vs default", "parameter_delta": qv_params-default_params,
    "parameter_reduction_pct": 100*(default_params-qv_params)/default_params if default_params else np.nan,
    "training_seconds_delta": qv_time-default_time,
    "training_time_reduction_pct": 100*(default_time-qv_time)/default_time if default_time else np.nan,
}])
print("Model quality, trained parameters, and training-only time:")
display(metrics_df.round(6))
print("Absolute score differences and relative percent changes:")
display(gains_df.round(6))
print("Q/V efficiency comparison (positive reduction means fewer parameters / less training time):")
display(efficiency_df.round(4))

predictions_df = pd.DataFrame({"test_index": range(EVAL_SAMPLE_SIZE),
    "dialogue": test_rows["dialogue"], "reference_summary": references,
    **{name + "_summary": preds for name, preds in predictions_by_model.items()}})
display(predictions_df)
print(f"Evaluated {len(references)} rows representing {len(set(test_rows['dialogue']))} unique dialogues.")

Model quality, trained parameters, and training-only time:


,model,parameters_updated,fine_tune_seconds,fine_tune_minutes,target_modules,rouge1,rouge2,rougeL,rougeLsum
0,vanilla_phi2,0,0.000000,0.000000,none,0.279663,0.099322,0.202289,0.213901
1,default_lora,20971520,2808.197763,46.803296,"q_proj, k_proj, v_proj, dense",0.434318,0.154607,0.362120,0.361928
2,qv_only_lora,10485760,2701.220444,45.020341,"q_proj, v_proj",0.418922,0.127022,0.312632,0.314277


Absolute score differences and relative percent changes:


,model,compared_with,metric,score,comparator_score,absolute_delta,relative_delta_pct
0,default_lora,vanilla_phi2,rouge1,0.434318,0.279663,0.154655,55.300382
1,default_lora,vanilla_phi2,rouge2,0.154607,0.099322,0.055285,55.662882
2,default_lora,vanilla_phi2,rougeL,0.362120,0.202289,0.159831,79.011124
3,default_lora,vanilla_phi2,rougeLsum,0.361928,0.213901,0.148026,69.203035
4,qv_only_lora,vanilla_phi2,rouge1,0.418922,0.279663,0.139259,49.795463
5,qv_only_lora,vanilla_phi2,rouge2,0.127022,0.099322,0.027701,27.889808
6,qv_only_lora,vanilla_phi2,rougeL,0.312632,0.202289,0.110343,54.547055
7,qv_only_lora,vanilla_phi2,rougeLsum,0.314277,0.213901,0.100376,46.926130
8,qv_only_lora,default_lora,rouge1,0.418922,0.434318,-0.015395,-3.544691
9,qv_only_lora,default_lora,rouge2,0.127022,0.154607,-0.027585,-17.841809


Q/V efficiency comparison (positive reduction means fewer parameters / less training time):


,comparison,parameter_delta,parameter_reduction_pct,training_seconds_delta,training_time_reduction_pct
0,QV vs default,-10485760,50.0,-106.9773,3.8095


,test_index,dialogue,reference_summary,vanilla_phi2_summary,default_lora_summary,qv_only_lora_summary
0,0,"#Person1#: Ms. Dawson, I need you to take a di...",Ms. Dawson helps #Person1# to write a memo to ...,"Person 1: Ms. Dawson, I need you to take a dic...",#Person1# asks Ms. Dawson to take a dictation ...,#Person1# tells Ms. Dawson to take a dictation...
1,1,"#Person1#: Ms. Dawson, I need you to take a di...",In order to prevent employees from wasting tim...,"Person 1: Ms. Dawson, I need you to take a dic...",#Person1# tells Ms. Dawson to take a dictation...,#Person1# asks Ms. Dawson to take a dictation ...
2,2,"#Person1#: Ms. Dawson, I need you to take a di...",Ms. Dawson takes a dictation for #Person1# abo...,"Person 1: Ms. Dawson, I need you to take a dic...",#Person1# asks Ms. Dawson to take a dictation ...,#Person1# tells Ms. Dawson to take a dictation...
3,3,#Person1#: You're finally here! What took so l...,#Person2# arrives late because of traffic jam....,Person1 and Person2 are discussing the traffic...,#Person2# got stuck in traffic again and #Pers...,#Person2# got stuck in traffic and #Person1# s...
4,4,#Person1#: You're finally here! What took so l...,#Person2# decides to follow #Person1#'s sugges...,Person1 and Person2 are discussing the traffic...,#Person2# got stuck in traffic again and #Pers...,#Person2# got stuck in traffic and #Person1# s...
5,5,#Person1#: You're finally here! What took so l...,#Person2# complains to #Person1# about the tra...,Person1 and Person2 are discussing the traffic...,#Person2# got stuck in traffic again and #Pers...,#Person2# got stuck in traffic and #Person1# s...
6,6,"#Person1#: Kate, you never believe what's happ...",#Person1# tells Kate that Masha and Hero get d...,Kate informed that Masha and Hero are getting ...,Kate tells #Person1# that Masha and Hero are g...,Kate tells #Person1# that Masha and Hero are g...
7,7,"#Person1#: Kate, you never believe what's happ...",#Person1# tells Kate that Masha and Hero are g...,Kate informed that Masha and Hero are getting ...,Kate tells #Person1# that Masha and Hero are g...,Kate and Masha are getting divorced. Kate is s...
8,8,"#Person1#: Kate, you never believe what's happ...",#Person1# and Kate talk about the divorce betw...,Kate informed that Masha and Hero are getting ...,Kate tells #Person1# that Masha and Hero are g...,Kate tells Masha that Masha and Hero are getti...
9,9,"#Person1#: Happy Birthday, this is for you, Br...",#Person1# and Brian are at the birthday party ...,"Person1 and Person2 are at a party, and Person...",Brian's birthday party is going well. #Person1...,#Person1# surprises Brian with a birthday part...


Evaluated 10 rows representing 4 unique dialogues.


## 8. Unified results

In [ ]:
print("Default LoRA compared with vanilla Phi-2")
for metric in METRICS:
    base = recomputed_scores["vanilla_phi2"][metric]
    score = recomputed_scores["default_lora"][metric]
    print(f"  {metric}: {base:.6f} -> {score:.6f}; delta {score-base:+.6f}")
print(f"  Default fine-tuning: {default_time:.2f} seconds ({default_time/60:.2f} minutes)")
print(f"  Default parameters updated: {default_params:,}")
print("\nQ/V-only LoRA compared with vanilla and assignment-default LoRA")
print(f"  Q/V parameters updated: {qv_params:,} (default: {default_params:,}; vanilla: 0)")
for metric in METRICS:
    qv = recomputed_scores["qv_only_lora"][metric]
    vanilla = recomputed_scores["vanilla_phi2"][metric]
    full = recomputed_scores["default_lora"][metric]
    print(f"  {metric}: Q/V {qv:.6f}; delta vs vanilla {qv-vanilla:+.6f}; delta vs default {qv-full:+.6f}")
print(f"  Q/V fine-tuning: {qv_time:.2f} seconds ({qv_time/60:.2f} minutes)")
print(f"  Q/V minus default training time: {qv_time-default_time:+.2f} seconds")
print("  Vanilla is inference-only; its fine-tuning time is 0 seconds.")

Default LoRA compared with vanilla Phi-2
  rouge1: 0.279663 -> 0.434318; delta +0.154655
  rouge2: 0.099322 -> 0.154607; delta +0.055285
  rougeL: 0.202289 -> 0.362120; delta +0.159831
  rougeLsum: 0.213901 -> 0.361928; delta +0.148026
  Default fine-tuning: 2808.20 seconds (46.80 minutes)
  Default parameters updated: 20,971,520

Q/V-only LoRA compared with vanilla and assignment-default LoRA
  Q/V parameters updated: 10,485,760 (default: 20,971,520; vanilla: 0)
  rouge1: Q/V 0.418922; delta vs vanilla +0.139259; delta vs default -0.015395
  rouge2: Q/V 0.127022; delta vs vanilla +0.027701; delta vs default -0.027585
  rougeL: Q/V 0.312632; delta vs vanilla +0.110343; delta vs default -0.049488
  rougeLsum: Q/V 0.314277; delta vs vanilla +0.100376; delta vs default -0.047651
  Q/V fine-tuning: 2701.22 seconds (45.02 minutes)
  Q/V minus default training time: -106.98 seconds
  Vanilla is inference-only; its fine-tuning time is 0 seconds.


## 9. Interpretation and limitations

The broader configuration has higher ROUGE on all four reported metrics and doubles the number of adapter parameters relative to Q/V-only. Training duration was similar (~46.8 vs ~45.0 minutes). However, evaluation on just ten examples is highly noisy; This exact notebook was ran in separate sessions and their baseline scores were not identical. This is an exploratory ablation, not a statistically robust benchmark.

**Next steps:** evaluate the full held-out set, standardize deterministic generation, repeat across seeds, inspect hallucinations manually, and profile peak GPU memory.

**Parameter-count caveat:** the original baseline requires_grad tally reported ~263M trainable parameters even though it was used only for inference. This is not the number of parameters updated during baseline evaluation. The historical table reports adapter parameters (0 for baseline) instead.